# 03 — Fact Generation

This notebook is the actual knowledge base of the RAG system: every aggregation we compute here becomes one retrievable fact sentence in Notebook 4. The design rule throughout is simple — **never generate a fact from fewer than 30 respondents**, and always carry the sample size in the sentence itself, so the LLM (and the person reading the answer) can judge how much to trust it.

In [10]:
import pandas as pd
from collections import Counter

DATA_DIR = "data"
df = pd.read_csv(f"{DATA_DIR}/results.txt", low_memory=False)

MIN_N = 30
facts = []

def add_fact(text, category, n):
    facts.append({"text": text, "category": category, "n": n})

## Setup: cleaned salary, experience buckets, allowlists

Reusing the salary filter from Notebook 2, plus two new things: experience buckets from `WorkExp` (capped at 50 years — a handful of respondents entered 60-100, clearly placeholder answers), and an allowlist of DevTypes/countries with enough respondents to be worth generating facts about at all.

In [11]:
df["comp_clean"] = df["ConvertedCompYearly"].where(
    (df["ConvertedCompYearly"] >= 1000) & (df["ConvertedCompYearly"] <= 2_000_000)
)

def bucket_exp(years):
    if pd.isna(years) or years > 50:
        return None
    if years <= 2: return "0-2 years"
    if years <= 5: return "3-5 years"
    if years <= 10: return "6-10 years"
    if years <= 20: return "11-20 years"
    return "21-50 years"

df["exp_bucket"] = df["WorkExp"].apply(bucket_exp)

exclude_devtypes = {"Other (please specify):", "Retired"}
devtype_counts = df["DevType"].value_counts()
devtype_allowlist = [d for d in devtype_counts.index if d not in exclude_devtypes and devtype_counts[d] >= 100]

top_countries = df["Country"].value_counts().head(15).index.tolist()

def parse_multiselect(series):
    return series.dropna().astype(str).apply(lambda x: [v.strip() for v in x.split(";") if v.strip()])

print(f"{len(devtype_allowlist)} roles and {len(top_countries)} countries pass the allowlist")

29 roles and 15 countries pass the allowlist


## Salary facts

Four cuts: by role, by role × country (only where both cross to a big enough sample), by experience bucket, and by country overall. The role × country cut is where most of our facts come from — it's also exactly the shape of question your own example targeted back in the pitch (a specific role, in a specific country).

In [12]:
for role in devtype_allowlist:
    sub = df.loc[df["DevType"] == role, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"The median annual salary for a {role} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_role", len(sub))

for role in devtype_allowlist:
    for country in top_countries:
        sub = df.loc[(df["DevType"] == role) & (df["Country"] == country), "comp_clean"].dropna()
        if len(sub) >= MIN_N:
            add_fact(f"In {country}, the median annual salary for a {role} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                      "salary_by_role_country", len(sub))

for bucket in ["0-2 years","3-5 years","6-10 years","11-20 years","21-50 years"]:
    sub = df.loc[df["exp_bucket"] == bucket, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers with {bucket} of professional experience report a median annual salary of \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_experience", len(sub))

for country in top_countries:
    sub = df.loc[df["Country"] == country, "comp_clean"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"The median annual salary in {country} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
                  "salary_by_country", len(sub))

print(f"Salary facts so far: {len(facts)}")

<>:4: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:23: SyntaxWarning: invalid escape sequence '\$'
<>:4: SyntaxWarning: invalid escape sequence '\$'
<>:11: SyntaxWarning: invalid escape sequence '\$'
<>:17: SyntaxWarning: invalid escape sequence '\$'
<>:23: SyntaxWarning: invalid escape sequence '\$'
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8104\1676493374.py:4: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"The median annual salary for a {role} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8104\1676493374.py:11: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"In {country}, the median annual salary for a {role} is \${int(sub.median()):,}, based on {len(sub)} respondents.",
C:\Users\ASUS\AppData\Local\Temp\ipykernel_8104\1676493374.py:17: SyntaxWarning: invalid escape sequence '\$'
  add_fact(f"Developers

Salary facts so far: 151


## Technology adoption facts

Overall top tools for the five biggest multi-select columns, plus top languages broken down by role — the same `parse_multiselect` helper from Notebook 2, just counted per group instead of once.

In [13]:
tech_columns = {
    "LanguageHaveWorkedWith": "language/technology",
    "DatabaseHaveWorkedWith": "database",
    "PlatformHaveWorkedWith": "cloud/platform tool",
    "WebframeHaveWorkedWith": "web framework",
    "AIModelsHaveWorkedWith": "AI model/LLM",
}

def top_n_multiselect(series, n=5):
    parsed = parse_multiselect(series)
    counts = Counter()
    for row in parsed:
        counts.update(row)
    return counts.most_common(n), len(parsed)

for col, label in tech_columns.items():
    top5, total = top_n_multiselect(df[col], n=5)
    if total >= MIN_N:
        for tech, c in top5:
            add_fact(f"{c/total*100:.0f}% of developers used {tech} as their {label} in the past year, based on {total} respondents.",
                      f"tech_adoption_overall_{col}", total)

for role in devtype_allowlist:
    sub_series = df.loc[df["DevType"] == role, "LanguageHaveWorkedWith"]
    top3, total = top_n_multiselect(sub_series, n=3)
    if total >= MIN_N:
        for lang, c in top3:
            add_fact(f"Among {role}s, {c/total*100:.0f}% used {lang} in the past year, based on {total} respondents.",
                      "tech_by_role", total)

print(f"Total facts so far: {len(facts)}")

Total facts so far: 263


## Job satisfaction and AI-perception facts

Satisfaction by remote-work status and by role, then the two AI-specific cuts: sentiment by experience level, and threat perception by role — the angle that ties this most directly back to the project's actual theme.

In [14]:
for remote_status in df["RemoteWork"].dropna().unique():
    sub = df.loc[df["RemoteWork"] == remote_status, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"Developers working '{remote_status}' report an average job satisfaction of {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_remote", len(sub))

for role in devtype_allowlist:
    sub = df.loc[df["DevType"] == role, "JobSat"].dropna()
    if len(sub) >= MIN_N:
        add_fact(f"{role}s report an average job satisfaction of {sub.mean():.1f} out of 10, based on {len(sub)} respondents.",
                  "satisfaction_by_role", len(sub))

for bucket in ["0-2 years","3-5 years","6-10 years","11-20 years","21-50 years"]:
    sub = df.loc[df["exp_bucket"] == bucket, "AISent"].dropna()
    if len(sub) >= MIN_N:
        fav_pct = sub.isin(["Favorable","Very favorable"]).mean() * 100
        add_fact(f"Among developers with {bucket} of experience, {fav_pct:.0f}% have a favorable view of AI tools, based on {len(sub)} respondents.",
                  "ai_sentiment_by_experience", len(sub))

for role in devtype_allowlist:
    sub = df.loc[df["DevType"] == role, "AIThreat"].dropna()
    if len(sub) >= MIN_N:
        yes_pct = (sub == "Yes").mean() * 100
        add_fact(f"{yes_pct:.0f}% of {role}s see AI as a threat to their current job, based on {len(sub)} respondents.",
                  "ai_threat_by_role", len(sub))

print(f"Total facts: {len(facts)}")
print(pd.Series([f['category'] for f in facts]).value_counts())

Total facts: 330
salary_by_role_country                          102
tech_by_role                                     87
ai_threat_by_role                                29
salary_by_role                                   29
satisfaction_by_role                             28
salary_by_country                                15
tech_adoption_overall_DatabaseHaveWorkedWith      5
tech_adoption_overall_LanguageHaveWorkedWith      5
salary_by_experience                              5
tech_adoption_overall_PlatformHaveWorkedWith      5
tech_adoption_overall_AIModelsHaveWorkedWith      5
tech_adoption_overall_WebframeHaveWorkedWith      5
satisfaction_by_remote                            5
ai_sentiment_by_experience                        5
Name: count, dtype: int64


## Save the facts table

This file is the direct input to Notebook 4 — every row becomes one embedded, searchable fact.

In [15]:
facts_df = pd.DataFrame(facts)
facts_df.to_csv(f"{DATA_DIR}/facts.csv", index=False)
print(f"Saved {len(facts_df)} facts to {DATA_DIR}/facts.csv")
facts_df.sample(5, random_state=1)[["text","n"]]

Saved 330 facts to data/facts.csv


,text,n
59,"In United States of America, the median annual...",68
267,Developers working 'Your choice (very flexible...,2848
180,"Among Developer, back-ends, 57% used JavaScrip...",4695
192,"Among Developer, desktop or enterprise applica...",1489
284,"Developer, game or graphicss report an average...",301


## What we built

- **330 facts** across salary (by role, role×country, experience, country), tech adoption (overall and by role), job satisfaction (by remote status and role), and AI sentiment/threat perception
- Every fact enforces a minimum sample size of 30 and states its N explicitly — no fact hides how confident it should be
- `facts.csv` is now the entire knowledge base the chatbot can draw on — if a question can't be answered from one of these 330 sentences, the retrieval step in Notebook 5 simply won't find a good match, which is the correct, honest behavior for a RAG system rather than letting the LLM guess

**Next:** Notebook 4 — embed all 330 facts and build the FAISS index.